In [14]:
#1. Install libraries
!pip install -q -U transformers accelerate bitsandbytes huggingface_hub
!pip install -q pandas==2.2.3
!pip install -U bitsandbytes>=0.46.1

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 88.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 793.2/793.2 kB 33.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 117.7 MB/s eta 0:00:00


In [15]:
#2. Login to Hugging Face
#Llama may require you to accept Meta's license/access conditions on Hugging Face first.
from huggingface_hub import notebook_login

notebook_login()

#You will be asked for your Hugging Face access token.

In [22]:
#3. Import libraries
import torch
import time
import pandas as pd

#Mistral 7B will be very slow and memory-heavy on CPU.
#Enable GPU in Google Colab, In Colab, go to:
# Runtime → Change runtime type → Hardware accelerator → GPU → Save

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

GPU available: True
GPU: Tesla T4


In [23]:
#4. Define the three LLMs
#The Qwen model card specifically recommends current versions of
#Transformers and demonstrates use with AutoModelForCausalLM, AutoTokenizer, and chat templates.

models = {
    #"Llama": "meta-llama/Llama-3.2-1B-Instruct",

    "Mistral": "mistralai/Mistral-7B-Instruct-v0.3",

    "Qwen": "Qwen/Qwen2.5-1.5B-Instruct"
}

models

{'Mistral': 'mistralai/Mistral-7B-Instruct-v0.3',
 'Qwen': 'Qwen/Qwen2.5-1.5B-Instruct'}

In [24]:
#5. Create an evaluation dataset
#For demonstration, we will use questions where the expected answer is easy to verify.

questions = [
    {
        "question": "What is the capital of France?",
        "expected": "Paris"
    },
    {
        "question": "What planet is known as the Red Planet?",
        "expected": "Mars"
    },
    {
        "question": "Who developed the theory of relativity?",
        "expected": "Einstein"
    },
    {
        "question": "What is 15 multiplied by 8?",
        "expected": "120"
    },
    {
        "question": "What gas do plants absorb from the atmosphere?",
        "expected": "carbon dioxide"
    }
]

questions


[{'question': 'What is the capital of France?', 'expected': 'Paris'},
 {'question': 'What planet is known as the Red Planet?', 'expected': 'Mars'},
 {'question': 'Who developed the theory of relativity?',
  'expected': 'Einstein'},
 {'question': 'What is 15 multiplied by 8?', 'expected': '120'},
 {'question': 'What gas do plants absorb from the atmosphere?',
  'expected': 'carbon dioxide'}]

In [25]:
#6. Function to load a model
#We will use 4-bit quantization for Mistral because the 7B model is much larger.
def load_model(model_name, model_label):

    print(f"\nLoading {model_label}...")

    tokenizer = AutoTokenizer.from_pretrained(model_name)

    # Mistral is larger, so use 4-bit quantization
    if model_label == "Mistral":

        quantization_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16
        )

        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            device_map="auto",
            quantization_config=quantization_config
        )

    else:

        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            device_map="auto",
            torch_dtype="auto"
        )

    return tokenizer, model

In [26]:
#7. Function to ask the LLM a question
#Notice that we use the model's chat template. This is important because
#different LLM families use different prompt formats.
def generate_answer(model, tokenizer, question):

    messages = [
        {
            "role": "system",
            "content": "Answer the question briefly and accurately."
        },
        {
            "role": "user",
            "content": question
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    start_time = time.time()

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=50,
            do_sample=False
        )

    end_time = time.time()

    # Remove input prompt tokens
    generated_tokens = output[0][inputs["input_ids"].shape[1]:]

    answer = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    latency = end_time - start_time

    return answer.strip(), latency

In [36]:
#!pip install -U accelerate>=1.2.0
!pip install -U bitsandbytes>=0.46.1

#8. Evaluate all three models
#This is the main evaluation loop.
!pip install -U bitsandbytes>=0.46.1

results = []

for model_label, model_name in models.items():

    tokenizer, model = load_model(
        model_name,
        model_label
    )

    print(f"\nTesting {model_label}")
    print("=" * 60)

    for item in questions:

        question = item["question"]
        expected = item["expected"]

        answer, latency = generate_answer(
            model,
            tokenizer,
            question
        )

        # Simple automatic correctness check
        correct = expected.lower() in answer.lower()

        results.append({
            "Model": model_label,
            "Question": question,
            "Expected Answer": expected,
            "Model Answer": answer,
            "Correct": correct,
            "Latency (seconds)": round(latency, 2)
        })

        print("\nQuestion:", question)
        print("Answer:", answer)
        print("Correct:", correct)
        print("Time:", round(latency, 2), "seconds")

    # Remove current model from GPU memory
    del model
    del tokenizer

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


Loading Mistral...


ImportError: Using `bitsandbytes` 4-bit quantization requires bitsandbytes: `pip install -U bitsandbytes>=0.46.1`

In [ ]:
#9. View complete results
df = pd.DataFrame(results)

print(df)

#10. Calculate accuracy
accuracy = (
    df.groupby("Model")["Correct"]
    .mean()
    .mul(100)
    .reset_index()
)

accuracy.columns = [
    "Model",
    "Accuracy (%)"
]

print(accuracy)

#11. Calculate average response time
latency = (
    df.groupby("Model")["Latency (seconds)"]
    .mean()
    .reset_index()
)

latency.columns = [
    "Model",
    "Average Latency (sec)"
]

print(latency)

